# synth_data_gen.ipynb

Sinh du lieu NER tong hop (BANGIAO §3-§6). Chay tren Kaggle GPU T4 x2, Internet ON.

**Datasets can add:** KB files tu repo (fakeer/kb/)

**Output:** `output_json/*.json` (schema: `[{text, type, position:[start,end]}]`)

In [ ]:
import os, sys, subprocess
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

REPO = 'https://github.com/Khanhhh239/fakeer.git'
if not os.path.exists('fakeer'):
    subprocess.run(['git', 'clone', '-q', REPO], check=True)
subprocess.run(['git', '-C', 'fakeer', 'pull', '-q', 'origin', 'main'])

sys.path.insert(0, 'fakeer/src')
import glob, json, random, re, time
os.makedirs('/kaggle/working/output_json', exist_ok=True)
print('Setup xong')

In [ ]:
import torch, inspect
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

_ngpu = max(1, torch.cuda.device_count())
print(f'{_ngpu} GPU kha dung')

def _mk(model, tp, util):
    want = dict(model=model, dtype='float16', max_model_len=3072,
                gpu_memory_utilization=util, tensor_parallel_size=tp,
                enforce_eager=True, disable_custom_all_reduce=True)
    ok = set(inspect.signature(LLM.__init__).parameters)
    want = {k: v for k, v in want.items() if k in ok}
    return LLM(**want)

LLM_MODEL = 'Qwen/Qwen3-8B'
_plans = [(LLM_MODEL, _ngpu, 0.90), (LLM_MODEL, _ngpu, 0.80),
          (LLM_MODEL, _ngpu, 0.70), ('Qwen/Qwen2.5-7B-Instruct', _ngpu, 0.85)]
llm = None
for _m, _tp, _u in _plans:
    try:
        print(f'Thu {_m} | TP={_tp} | util={_u}')
        llm = _mk(_m, _tp, _u)
        LLM_MODEL = _m
        break
    except Exception as e:
        print('  x', type(e).__name__, str(e)[:120])
assert llm is not None, 'Khong nap duoc LLM nao'
qtok = AutoTokenizer.from_pretrained(LLM_MODEL)
print('DUNG:', LLM_MODEL)

In [ ]:
from synth_source import load_chandoan, load_trieuchung, load_thuoc, _load_txt

chandoan_pool = load_chandoan()
trieuchung_pool = load_trieuchung()
thuoc_pool = load_thuoc()
xn_names = _load_txt('xetnghiem_ten.txt')
am_thuoc = _load_txt('am_thuoc_gia.txt')
am_xetnghiem = _load_txt('am_xetnghiem_gia.txt')
heading_ls = _load_txt('heading_lamsang.txt')
heading_hd = _load_txt('heading_hoidap.txt')

print(f'Chan doan: {len(chandoan_pool)}, Trieu chung: {len(trieuchung_pool)}')
print(f'Thuoc: {len(thuoc_pool)}, XN ten: {len(xn_names)}')
print(f'Am thuoc: {len(am_thuoc)}, Am XN: {len(am_xetnghiem)}')

In [ ]:
PROMPT_T1_TEMPLATE = 'CHI TRA LOI BANG TIENG VIET. Khong dung chu Han, chu Trung Quoc, tieng Anh.\n\nBan la bac si Viet Nam. Voi chan doan duoc cho, liet ke cac khai niem y khoa thuong di kem trong benh an.\n\nDINH DANG: moi dong mot muc, dung dang MA|noi dung\n  TC = trieu chung nguoi benh cam nhan hoac bac si quan sat duoc\n  TH = ten thuoc dieu tri\n  TX = ten xet nghiem / tham do / thu thuat chan doan\n\nSO LUONG: 5 dong TC, 3 dong TH, 3 dong TX. Tong dung 11 dong.\n\nQUY TAC:\n1. Chi ghi TEN, khong ghi dong tu di kem.\n2. TH phai la ten thuoc CO THAT. Khong bia.\n   DUNG: TH|paracetamol, TH|khang sinh, TH|corticoid\n3. TX phai la TEN mot xet nghiem cu the, KHONG phai mo ta cach lam.\n   DUNG: TX|cong thuc mau, TX|sieu am o bung\n4. TC phai la dieu nguoi benh CAM THAY hoac bac si THAY.\n   DUNG: TC|sot cao, TC|dau vung thuong vi\n5. Moi muc phai lien quan TRUC TIEP toi chan doan da cho.\n6. Moi muc 1-5 tu, viet nhu bac si ghi benh an.\n7. KHONG chu thich ten nuoc ngoai trong ngoac.\n8. Viet xong 11 dong thi DUNG. Khong giai thich.\n\nBAY GIO LAM VOI CHAN DOAN: "{diagnosis}"\n\nNhac lai: chi tieng Viet. Khong chu Han. Khong tieng Anh.'

def build_t1_prompt(diagnosis):
    messages = [{'role': 'user',
                 'content': PROMPT_T1_TEMPLATE.format(diagnosis=diagnosis)}]
    p = qtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Tat thinking Qwen3
    if '/no_think' not in p:
        p += '/no_think\n'
    return p

def parse_t1_response(text):
    tc, th, tx = [], [], []
    for line in text.strip().split('\n'):
        line = line.strip()
        if '|' not in line: continue
        code_, _, content = line.partition('|')
        code_ = code_.strip().upper(); content = content.strip()
        if not content or len(content) < 2: continue
        if code_ == 'TC': tc.append(content)
        elif code_ == 'TH': th.append(content)
        elif code_ == 'TX': tx.append(content)
    return {'tc': tc[:5], 'th': th[:3], 'tx': tx[:3]}

# Phan tang: moi chuong ICD chon ~5 chan doan -> ~100 chan doan tong
from collections import defaultdict
by_ch = defaultdict(list)
for d in chandoan_pool:
    by_ch[d['chapter']].append(d)
sample_diag = []
for ch, items in by_ch.items():
    sample_diag.extend(random.sample(items, min(5, len(items))))
random.shuffle(sample_diag)
print(f'Chan doan mau: {len(sample_diag)}')

t1_prompts = [build_t1_prompt(d['term']) for d in sample_diag]
t0 = time.time()
t1_outs = llm.generate(t1_prompts, SamplingParams(temperature=0.3, max_tokens=300))
print(f'T1 xong {len(t1_outs)} kich ban trong {time.time()-t0:.0f}s')

scenarios = []
for d, out in zip(sample_diag, t1_outs):
    parsed = parse_t1_response(out.outputs[0].text)
    if len(parsed['tc']) >= 3:
        scenarios.append({'diagnosis': d, 'scenario': parsed})
print(f'Kich ban hop le: {len(scenarios)} / {len(sample_diag)}')
print('Vi du:', scenarios[0] if scenarios else None)

In [ ]:
from synth_anchor import anchor_all, validate_document

PROMPT_T2B = "CHI VIET BANG TIENG VIET. Khong dung chu Han, chu Trung Quoc. Duoc phep dung thuat ngu y khoa chuan viet bang chu La-tinh.\n\nBan la bien tap vien chuyen muc tu van suc khoe cua mot trang web y te Viet Nam.\nViet MOT BAI tu van hoan chinh.\n\nBai gom dung cac phan sau, viet lien mach:\n\nCau hoi tu nguoi dung:\n(4 den 6 cau, nguoi benh tu ke: hoan canh, kho chiu ra sao, lo lang gi, roi hoi)\n\nCau tra loi cua bac si:\nChao ban,\n1. {tieu_de_1}\n(4 den 6 cau)\n2. {tieu_de_2}\n(4 den 6 cau)\n3. {tieu_de_3}\n(4 den 6 cau)\n4. {tieu_de_4}\n(4 den 6 cau)\nTran trong!\n\nCAC CUM SAU PHAI XUAT HIEN NGUYEN VAN trong bai, khong sua mot chu:\n{danh_sach_cum}\n- {bait_thuoc}\n- {bait_xetnghiem}\n\nQUY TAC:\n- Tong bai 500 den 700 tu.\n- Mo phong loi go: dinh lien {n_glue} cho (bo dau cach).\n- Chen dau sao {n_mask} cho, vi du 'Khang sinh nhom ***'.\n- Chi ap dung loi o phan van xuoi, KHONG duong vao cac cum bat buoc va bait.\n- Cam cau tran an rong. Cam viet doi thoai qua lai.\n- Khong dung gach dau dong trong phan tra loi.\n\nNhac lai: viet tieng Viet, thuoc/xet nghiem giu ten quoc te chuan."

TYPE_MAP = {
    'CHAN_DOAN': 'CHAN_DOAN', 'TRIEU_CHUNG': 'TRIEU_CHUNG',
    'THUOC': 'THUOC', 'TEN_XET_NGHIEM': 'TEN_XET_NGHIEM',
    'KET_QUA_XET_NGHIEM': 'KET_QUA_XET_NGHIEM',
}

def build_t2b_prompt(scenario):
    sc = scenario['scenario']
    diag = scenario['diagnosis']['term']
    ents = [(diag, 'CHAN_DOAN')]
    for t in sc['tc']: ents.append((t, 'TRIEU_CHUNG'))
    for t in sc['th']: ents.append((t, 'THUOC'))
    for t in sc['tx']: ents.append((t, 'TEN_XET_NGHIEM'))
    for xn in random.sample(xn_names, min(3, len(xn_names))):
        ents.append((xn, 'TEN_XET_NGHIEM'))
    random.shuffle(ents)
    ents = ents[:15]
    bt = random.choice(am_thuoc)
    bx = random.choice(am_xetnghiem)
    n_glue = random.randint(1, 5)
    n_mask = random.randint(1, 3)
    tds_pool = [
        f'{diag} la benh gi', f'Trieu chung cua {diag}',
        f'Nguyen nhan gay {diag}', f'Dieu tri {diag}',
        f'Phong ngua {diag}', 'Khi nao can gap bac si'
    ]
    tds = random.sample(tds_pool, 4)
    danh_sach = '\n'.join(f'- {s}' for s, _ in ents)
    prompt_text = PROMPT_T2B.format(
        danh_sach_cum=danh_sach, bait_thuoc=bt, bait_xetnghiem=bx,
        n_glue=n_glue, n_mask=n_mask,
        tieu_de_1=tds[0], tieu_de_2=tds[1],
        tieu_de_3=tds[2], tieu_de_4=tds[3],
    )
    messages = [{'role': 'user', 'content': prompt_text}]
    p = qtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if '/no_think' not in p:
        p += '/no_think\n'
    return p, ents, bt, bx

t2b_inputs = []
for sc in scenarios:
    p, req, bt, bx = build_t2b_prompt(sc)
    t2b_inputs.append((p, req, bt, bx, sc))

t2b_prompts = [x[0] for x in t2b_inputs]
t0 = time.time()
t2b_outs = llm.generate(t2b_prompts, SamplingParams(temperature=0.7, max_tokens=900))
print(f'T2B xong {len(t2b_outs)} bai trong {time.time()-t0:.0f}s')

In [ ]:
saved = 0
reject_log = {}

for i, (t2b_out, (_, req, bait_t, bait_x, sc)) in enumerate(
        zip(t2b_outs, t2b_inputs)):
    text = t2b_out.outputs[0].text.strip()
    if not text:
        reject_log['empty'] = reject_log.get('empty', 0) + 1
        continue

    ents = anchor_all(text, req, bait=[bait_t, bait_x])
    if ents is None:
        reject_log['anchor<60%'] = reject_log.get('anchor<60%', 0) + 1
        continue

    ok, reason = validate_document(
        text, ents, bait_thuoc=bait_t, bait_xetnghiem=bait_x,
        am_thuoc=am_thuoc, am_xetnghiem=am_xetnghiem,
    )
    if not ok:
        gate = reason.split(']')[0].lstrip('[')
        reject_log[gate] = reject_log.get(gate, 0) + 1
        continue

    out_ents = [{'text': e['text'], 'type': e['type'],
                 'position': e['position']} for e in ents]
    fname = f'{saved+1:04d}.json'
    with open(f'/kaggle/working/output_json/{fname}', 'w', encoding='utf-8') as f:
        json.dump(out_ents, f, ensure_ascii=False, indent=2)
    saved += 1

total_reject = sum(reject_log.values())
print(f'LUU: {saved} file, BO QUA: {total_reject}')
print('Ly do bo qua:', reject_log)
total_gen = saved + total_reject
if total_gen > 0:
    pct = total_reject / total_gen * 100
    if pct > 40:
        print(f'!!! {pct:.0f}% bi loai > 40% -- xem lai prompt/nguong')
    else:
        print(f'  Ty le bo qua: {pct:.0f}% (< 40%)')

In [ ]:
from synth_struct import build_blocks
from synth_source import gen_lab_pairs

def make_struct_entities(scenarios):
    result = []
    for sc in scenarios:
        s = sc['scenario']
        ents = [(sc['diagnosis']['term'], 'CHAN_DOAN')]
        for t in s['tc']: ents.append((t, 'TRIEU_CHUNG'))
        for t in s['th']: ents.append((t, 'THUOC'))
        for t in s['tx']: ents.append((t, 'TEN_XET_NGHIEM'))
        for full, ten, kq in gen_lab_pairs(3):
            ents.append((ten, 'TEN_XET_NGHIEM'))
            if kq: ents.append((kq, 'KET_QUA_XET_NGHIEM'))
        result.append(ents)
    return result

struct_ents_list = make_struct_entities(scenarios)
n_struct = max(20, int(saved * 10 // 65))
blocks = build_blocks(struct_ents_list, n_struct, heading_ls, heading_hd)

struct_saved = 0
struct_start = saved + 1
for text, ents in blocks:
    ok_ents = all(text[e['position'][0]:e['position'][1]] == e['text']
                  for e in ents)
    if not ok_ents: continue
    out_ents = [{'text': e['text'], 'type': e['type'],
                 'position': e['position']} for e in ents]
    fname = f'{struct_start + struct_saved:04d}.json'
    with open(f'/kaggle/working/output_json/{fname}', 'w', encoding='utf-8') as f:
        json.dump(out_ents, f, ensure_ascii=False, indent=2)
    struct_saved += 1

print(f'T2A: luu {struct_saved} khoi cau truc')
print(f'Tong file output_json/: {saved + struct_saved}')

In [ ]:
from collections import Counter
all_ents = []
fnames = sorted(os.listdir('/kaggle/working/output_json'))
for fn in fnames:
    if not fn.endswith('.json'): continue
    ents = json.load(open(f'/kaggle/working/output_json/{fn}', encoding='utf-8'))
    all_ents.extend(ents)

cnt = Counter(e['type'] for e in all_ents)
total = len(all_ents)
print(f'Tong thuc the: {total} trong {len(fnames)} file')
print('Phan bo loai:')
TARGET = {'CHAN_DOAN': 25, 'TRIEU_CHUNG': 33, 'THUOC': 16,
           'TEN_XET_NGHIEM': 15, 'KET_QUA_XET_NGHIEM': 11}
for t, tgt in sorted(TARGET.items(), key=lambda x: -x[1]):
    n = cnt.get(t, 0)
    pct = n / max(1, total) * 100
    flag = 'ok' if abs(pct - tgt) <= 15 else 'LECH'
    print(f'  {flag} {t}: {n} ({pct:.1f}%, muc tieu {tgt}%)')
print('\nSan sang dung lam training data cho encoder!')